# Strain2D: start here

This notebook tests the full workflow on **synthetic functions**, not real h-BN. Read `docs/SUPERVISOR_BRIEF.md` before interpreting results. Use the project environment as your Jupyter kernel.


In [ ]:
from pathlib import Path
import subprocess, sys, uuid
root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Open from the repository or notebooks directory"
run = root / "runs" / ("notebook-" + uuid.uuid4().hex[:8])
def cli(*args):
    subprocess.run([sys.executable, "-m", "strain2d", *map(str,args)], cwd=root, check=True)


In [ ]:
cli("demo-data", "--n", 400, "--out", root / "examples/SYNTHETIC_demo.csv")
cli("check", root / "examples/SYNTHETIC_demo.csv")
cli("train", "--data", root / "examples/SYNTHETIC_demo.csv", "--config", root / "configs/smoke.yaml", "--out", run)


In [ ]:
import json, pandas as pd
manifest = json.loads((run / "manifest.json").read_text())
print("Selected on validation:", manifest["selected_model"])
display(pd.read_csv(run / "metrics.csv"))


In [ ]:
cli("predict", "--run", run, "--points", root / "examples/prediction_points.csv", "--out", run / "predictions.csv")
display(pd.read_csv(run / "predictions.csv"))


In [ ]:
from IPython.display import Markdown, display, Image
display(Markdown((run / "report.md").read_text()))
display(Image(filename=str(run / "figures/bandgap_eV.png")))


In [ ]:
cli("interpret", "--run", run)
cli("sensitivity", "--run", run, "--points", root / "examples/prediction_points.csv", "--out", run / "sensitivity.csv")
cli("sample", "--n", 300, "--out", run / "candidates.csv")
cli("suggest", "--run", run, "--points", run / "candidates.csv", "--n", 10, "--out", run / "next_dft.csv")
display(pd.read_csv(run / "next_dft.csv"))


## Your real material

Follow `docs/HPC.md` to converge a reference and run a small VASP/phonon pilot. Then collect the dataset and use `configs/default.yaml` in a new run. Do not relabel the synthetic CSV as DFT. A material-specific trained model and independently validated predictions require real data.
